# **Deductive Logic in Language Models: Horizontal vs. Vertical Reasoning - Task 2 Training without CoT**

This notebook reproduces the training procedure for Task 2 without Chain-of-Thought (CoT) reasoning, as described in Section 7.2 of [D. Maltoni and M. Ferrara, *"Deductive Logic in Language Models: Horizontal vs. Vertical Reasoning"*, Machine Learning and Knowledge Extraction, 2026](https://doi.org/10.3390/make8070214).

The following code imports the modules and functions required to run the notebook.

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from dataset import GraphDataset_Without_CoT
from utils import train, get_loaders
from transformer_lens import HookedTransformer, HookedTransformerConfig

The following parameters configure the dataset and the nanoGPT architecture used in the experiment.

In [ ]:
# dataset configuration
dataset_config = {
    "n_states": 16,
    "dataset_file_name": "dataset_150k_remapped_without_CoT_paper.txt",                 
    "n_samples": 150_000,                                             
    "batch_size": 64,
}

# model configuration
model_config = {
    "n_layers": 6,
    "d_model": 128,
    "n_heads": 1,
    "d_mlp": 512,
    "d_head": 128,
}

init_seed = 72

Running the following cell properly initialises all random seeds to ensure reproducible results.

In [ ]:
random.seed(init_seed)
np.random.seed(init_seed)
torch.manual_seed(init_seed)
torch.cuda.manual_seed_all(init_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

The following code loads the dataset.

In [ ]:
dataset = GraphDataset_Without_CoT(dataset_config["n_states"], dataset_config["dataset_file_name"])

The following code creates the training and test data loaders using the batch size specified in the dataset configuration.

In [ ]:
train_loader, test_loader = get_loaders(dataset, dataset_config["batch_size"])

The following code configures and initialises the Transformer model.

In [ ]:
# setup model
cfg = HookedTransformerConfig(
    n_layers=model_config["n_layers"],
    d_model=model_config["d_model"],
    n_ctx=dataset.max_seq_length - 1,
    n_heads=model_config["n_heads"],
    d_mlp=model_config["d_mlp"],
    d_head=model_config["d_head"],
    d_vocab=len(dataset.idx2tokens),
    device="cuda",
    attention_dir="causal",
    act_fn="gelu",
)
model = HookedTransformer(cfg)

The following code trains the model without CoT and records its autoregressive accuracy on the training and test sets.

In [ ]:
# start training loop
train_acc_ar, test_acc_ar, _, _ = train(model,train_loader,test_loader,n_epochs=50,learning_rate=3e-4)

Running the cell below reproduces the plot showing the model accuracy obtained without CoT information.

In [ ]:
plt.plot(np.array(train_acc_ar)*100,label='Train')
plt.plot(np.array(test_acc_ar)*100,label='Val')
plt.xlabel('Epochs')
plt.ylabel('Sequence Accuracy (%)')
plt.title(f'Accuracy on Task 2 without CoT')
plt.legend()

plt.show()